In [12]:
#!pip install mne
import mne
import numpy as np
import pandas as pd
from mne.datasets.sleep_physionet.age import fetch_data

In [4]:
# Download data for 2 subjects only (enough to start, full dataset is huge)
subjects = [0]

recordings = fetch_data(subjects=subjects, recording=[1])

print(recordings)

Using default location ~/mne_data for PHYSIONET_SLEEP...
Creating /root/mne_data


  0%|                                              | 0.00/48.3M [00:00<?, ?B/s]

  0%|                                              | 0.00/4.62k [00:00<?, ?B/s]

Download complete in 04m18s (46.1 MB)
[['/root/mne_data/physionet-sleep-data/SC4001E0-PSG.edf', '/root/mne_data/physionet-sleep-data/SC4001EC-Hypnogram.edf']]


In [5]:
import mne

# Load raw EEG for first subject
raw = mne.io.read_raw_edf(recordings[0][0], preload=True)
annotations = mne.read_annotations(recordings[0][1])

raw.set_annotations(annotations)

print(raw.info)


Extracting EDF parameters from /root/mne_data/physionet-sleep-data/SC4001E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...


/tmp/ipykernel_41703/3811619814.py:4: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(recordings[0][0], preload=True)
/tmp/ipykernel_41703/3811619814.py:4: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(recordings[0][0], preload=True)
/tmp/ipykernel_41703/3811619814.py:4: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(recordings[0][0], preload=True)


Reading 0 ... 7949999  =      0.000 ... 79499.990 secs...
<Info | 8 non-empty values
 bads: []
 ch_names: EEG Fpz-Cz, EEG Pz-Oz, EOG horizontal, Resp oro-nasal, EMG ...
 chs: 7 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 50.0 Hz
 meas_date: 1989-04-24 16:13:00 UTC
 nchan: 7
 projs: []
 sfreq: 100.0 Hz
 subject_info: <subject_info | his_id: X, sex: 2, first_name: Female, last_name: 33yr>
>


/tmp/ipykernel_41703/3811619814.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw.set_annotations(annotations)


In [6]:

print(raw.ch_names)

['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']


In [7]:
# Keep only EEG channels
raw.pick_channels(['EEG Fpz-Cz', 'EEG Pz-Oz'])

# Bandpass filter 0.5 - 30 Hz (standard for sleep EEG)
raw.filter(0.5, 30.0)

print("Filtered. Channels:", raw.ch_names)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 661 samples (6.610 s)

Filtered. Channels: ['EEG Fpz-Cz', 'EEG Pz-Oz']


In [8]:
print("Sampling frequency:", raw.info['sfreq'])

Sampling frequency: 100.0


In [9]:
from mne.epochs import make_fixed_length_epochs

# Map sleep stage annotations to numbers
annotation_desc_2_event_id = {
    'Sleep stage W': 0,   # Wake
    'Sleep stage 1': 1,   # N1
    'Sleep stage 2': 2,   # N2
    'Sleep stage 3': 3,   # N3
    'Sleep stage 4': 3,   # N3 (same as stage 3)
    'Sleep stage R': 4,   # REM
}

events, event_id = mne.events_from_annotations(
    raw, event_id=annotation_desc_2_event_id, chunk_duration=30.0
)

print("Events shape:", events.shape)
print("Event IDs:", event_id)

# Create 30 second epochs
tmax = 30.0 - 1.0/raw.info['sfreq']
epochs = mne.Epochs(raw, events, event_id=event_id,
                    tmin=0.0, tmax=tmax,
                    baseline=None, preload=True)

print(epochs)

Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]
Events shape: (2650, 3)
Event IDs: {np.str_('Sleep stage 1'): 1, np.str_('Sleep stage 2'): 2, np.str_('Sleep stage 3'): 3, np.str_('Sleep stage 4'): 3, np.str_('Sleep stage R'): 4, np.str_('Sleep stage W'): 0}
Not setting metadata
2650 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 2650 events and 3000 original time points ...
0 bad epochs dropped
<Epochs | 2650 events (all good), 0 – 29.99 s (baseline off), ~121.3 MiB, data loaded,
 np.str_('Sleep stage 1'): 58
 np.str_('Sleep stage 2'): 250
 np.str_('Sleep stage 3'): 220
 np.str_('Sleep stage 4'): 220
 np.str_('Sleep stage R'): 125
 np.str_('Sleep stage W'): 1997>


In [10]:
# Get raw epoch data directly from epochs
X_raw = epochs.get_data()  # shape: (2650, 2, 3000)
print("Raw data shape:", X_raw.shape)

# Convert to microvolts
X_raw = X_raw * 1e6

# Reshape for CNN - needs (samples, timesteps, channels)
X_cnn = X_raw.transpose(0, 2, 1)  # (2650, 3000, 2)
y_cnn = epochs.events[:, 2]  # labels directly from epochs

print("CNN input shape:", X_cnn.shape)
print("Labels shape:", y_cnn.shape)

Raw data shape: (2650, 2, 3000)
CNN input shape: (2650, 3000, 2)
Labels shape: (2650,)


In [11]:
from sklearn.model_selection import train_test_split

X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(
    X_cnn, y_cnn, test_size=0.2, random_state=42, stratify=y_cnn
)

print("Train shape:", X_train_cnn.shape)
print("Test shape:", X_test_cnn.shape)

# Class weights instead of SMOTE for imbalance
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_cnn)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_cnn)
class_weights = dict(zip(classes, weights))
print("Class weights:", class_weights)

Train shape: (2120, 3000, 2)
Test shape: (530, 3000, 2)
Class weights: {np.int64(0): np.float64(0.26533166458072593), np.int64(1): np.float64(9.217391304347826), np.int64(2): np.float64(2.12), np.int64(3): np.float64(2.409090909090909), np.int64(4): np.float64(4.24)}


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

In [15]:
cnn_model = Sequential([
    Conv1D(filters=64, kernel_size=50, strides=6, activation='relu', input_shape=(3000, 2)),
    BatchNormalization(),
    MaxPooling1D(pool_size=8),
    Dropout(0.3),

    Conv1D(filters=128, kernel_size=8, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=4),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(5, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])


In [17]:
cnn_model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 492, 64)        │         6,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 492, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 61, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 61, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 54, 128)        │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 54, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 13, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 13, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1664)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       213,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 286,661 (1.09 MB)

 Trainable params: 286,277 (1.09 MB)

 Non-trainable params: 384 (1.50 KB)

In [19]:
history_cnn = cnn_model.fit(X_train_cnn, y_train_cnn,
                            epochs=50,
                            batch_size=32,
                            validation_split=0.2,
                            callbacks=[early_stop],
                            class_weight=class_weights,
                            verbose=1)

Epoch 1/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 7s 73ms/step - accuracy: 0.5932 - loss: 1.8768 - val_accuracy: 0.1014 - val_loss: 12.7800
Epoch 2/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 6s 111ms/step - accuracy: 0.8219 - loss: 0.9129 - val_accuracy: 0.1368 - val_loss: 4.2989
Epoch 3/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.8709 - loss: 0.7330 - val_accuracy: 0.8208 - val_loss: 0.6189
Epoch 4/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.8827 - loss: 0.6244 - val_accuracy: 0.8962 - val_loss: 0.3735
Epoch 5/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 7s 95ms/step - accuracy: 0.8880 - loss: 0.6757 - val_accuracy: 0.8113 - val_loss: 0.5829
Epoch 6/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - accuracy: 0.9139 - loss: 0.5320 - val_accuracy: 0.8939 - val_loss: 0.3262
Epoch 7/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 5s 68ms/step - accuracy: 0.9186 - loss: 0.4093 - val_accuracy: 0.9245 - val_loss: 0.2671
Epoch 8/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 6s 78ms/step - accuracy: 0.9404 - loss: 0.3289 - val_accuracy: 0.9245 -

In [20]:
y_pred_cnn = np.argmax(cnn_model.predict(X_test_cnn), axis=1)


17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


In [23]:
from sklearn.metrics import accuracy_score, classification_report

print("CNN Accuracy:", accuracy_score(y_test_cnn, y_pred_cnn))
print(classification_report(y_test_cnn, y_pred_cnn))

CNN Accuracy: 0.930188679245283
              precision    recall  f1-score   support

           0       1.00      0.97      0.99       399
           1       0.29      0.92      0.44        12
           2       0.91      0.84      0.88        50
           3       0.91      0.91      0.91        44
           4       0.86      0.48      0.62        25

    accuracy                           0.93       530
   macro avg       0.79      0.82      0.77       530
weighted avg       0.96      0.93      0.94       530

